In [1]:
import pandas as pd
import preprocessing.constants as c

In [2]:
df = pd.read_csv(f'{c.BASE_DIR}/baza_elm19/ELM19_info.csv', encoding='utf-8')

In [3]:
df.head()

,examination_id,patient_original_id,age_dec,patient_sex,institution_id,classification
0,20220411-162228-{4fe29d03-5025-46ca-8e14-c3b3f...,{d2bff7c3-8b81-4dfd-b4fe-4d9fe4c98a9e},56.583333,Male,TER_L,norm
1,20220510-150544-{cc1f09f8-3b30-40b4-91f8-4cd70...,{87ddf489-02db-4963-a966-b08709bbf636},38.000000,Male,KLU,patho
2,20220412-172841-{0881d6c7-8c23-4389-87c1-58c8e...,{3ec015e0-73f1-4ab4-9a3d-c01f353caa55},20.416667,Male,STG1,patho
3,20220530-182538-{85ec4ff0-fd03-4be2-a44b-99723...,{3b323fd5-59c0-4d05-a0d7-37e871d26df8},23.583333,Female,SRK,patho
4,20220512-152511-{61074308-a109-40b1-97cb-43dcb...,{238385d0-7169-42a6-b94f-6ddbc4e08746},60.583333,Male,SZC,patho


The head of original metadata file shows examination-ids as UUID v4 wrapped in a custom format with a timestamp and patient-ids as UUID v4. This i useful in hospital setting, but human-unreadable and not helping in classification pipeline.

That's why it is useful to map exam-id, patient-id, round age_dec to .2f, reduce Male and Female labels to classics M and F used in MNE and binarize classification to 0 (control, baseline, negative) and 1 (target, pathological, positive).

Update: i'll leave the exam-ids in UUID format. This enables then an easy matching between preprocessed matadata and reading EDFs into the pipeline.

Update: i'll leave the sb-ids in UUID format FOR NOW and clean it after processing & exclusions. Then, i'll also change hospital names to readible format.

In [8]:
clean_df_filename = f'{c.BASE_DIR}/magisterka/metadata/ELM19_metadata.csv'
clean_df = pd.DataFrame(columns=['sub_id', 'exam_id', 'age', 'female', 'site', 'pathology'])

In [9]:
df['institution_id']

0        TER_L
1          KLU
2         STG1
3          SRK
4          SZC
         ...  
55782     Z04O
55783     Z04O
55784     Z04O
55785     Z04O
55786     Z04O
Name: institution_id, Length: 55787, dtype: object

In [ ]:
stop = len(df) + 1
sex_map = {'Male': 0, 'Female': 1} # target is female
label_map = {'norm': 0, 'patho': 1} # target is pathology

# change UUID to readable format
clean_df['sub_id'] = df['patient_original_id'] # unique and non-unique ids
clean_df['exam_id'] = df['examination_id']


#clean_df['exam_id'] = [f"exam-{i:03d}" for i in range(1, stop)] # only unique ids

# round age in decimals to MNE's standard (and rational)
clean_df['age'] = df['age_dec'].round(1)

# map sex to M/F and label to 0/1
clean_df['female'] = df['patient_sex'].map(sex_map).astype(int)
clean_df['pathology'] = df['classification'].map(label_map).astype(int)

# copy institutions but with changed column name
clean_df['site'] = df['institution_id']

In [11]:
clean_df

,sub_id,exam_id,age,female,site,pathology
0,{d2bff7c3-8b81-4dfd-b4fe-4d9fe4c98a9e},20220411-162228-{4fe29d03-5025-46ca-8e14-c3b3f...,56.6,0,TER_L,0
1,{87ddf489-02db-4963-a966-b08709bbf636},20220510-150544-{cc1f09f8-3b30-40b4-91f8-4cd70...,38.0,0,KLU,1
2,{3ec015e0-73f1-4ab4-9a3d-c01f353caa55},20220412-172841-{0881d6c7-8c23-4389-87c1-58c8e...,20.4,0,STG1,1
3,{3b323fd5-59c0-4d05-a0d7-37e871d26df8},20220530-182538-{85ec4ff0-fd03-4be2-a44b-99723...,23.6,1,SRK,1
4,{238385d0-7169-42a6-b94f-6ddbc4e08746},20220512-152511-{61074308-a109-40b1-97cb-43dcb...,60.6,0,SZC,1
...,...,...,...,...,...,...
55782,{301c27a8-98d1-4130-b8f7-1f7fce39544f},20230227-224412-{42b0e675-b0f4-44dc-841e-d371a...,25.0,1,Z04O,1
55783,{e76dcd3d-aa25-480c-8a7d-6af413ad1e04},20230227-224412-{71d4a67d-5f7c-4b7f-bdf7-7f75f...,55.0,1,Z04O,0
55784,{724c1b9b-1154-476f-9fff-f176b509f895},20230227-223035-{fd275507-c177-4633-9dfe-1816b...,52.0,0,Z04O,1
55785,{12da4671-e811-45b9-940f-0432d6bf927a},20230227-223418-{dff64315-d77e-41a2-8125-a33ce...,27.8,0,Z04O,1


In [12]:
clean_df.to_csv(clean_df_filename, index = False)

Now, the file ELM19_metadata.csv can be ingested into ELM19_metadata_processor.